# L'effet domino dans le tunnel RER B / RER D

**Projet Python pour la data science — ENSAE 2A**

## Problématique

Entre *Châtelet-Les Halles* et *Gare du Nord*, le RER B et le RER D circulent sur **les mêmes voies**. Les usagers ont l'impression qu'un incident sur l'une des deux lignes se propage rapidement à l'autre.

**Hypothèse :** un incident sur le RER B (ou D) se propage à l'autre ligne à cause du tronçon commun, ce qui rend la régulation plus difficile et les retards plus persistants.

**Point de comparaison :** le **RER A**, dont le tronçon central (*Charles de Gaulle-Étoile* – *Nation*) est très fréquenté mais **ne partage pas ses voies** avec une autre ligne de RER. Il sert de référence (placebo).

## Plan

0. Préparation de l'environnement
1. Données : collecte, nettoyage et description
2. Axe 1 — Constat : B et D sont-ils plus en retard que A ?
3. Axe 2 — Contagion : les retards de B et D sont-ils liés ?
4. Axe 3 — Détection automatique des principaux incidents
5. Axe 4 — Modélisation économétrique : contagion et effet tunnel
6. Conclusion et limites

> Le code des analyses est dans le dossier `src/` (un fichier par axe). Ce notebook les exécute dans l'ordre et en interprète les résultats.

## 0. Préparation de l'environnement

Ce notebook s'exécute **depuis la racine du dépôt** avec *Restart & Run All*. Testé sur SSP Cloud (image Python / VSCode ou Jupyter).

Au premier lancement, les données sont **téléchargées automatiquement** depuis le bucket S3 public du SSP Cloud puis nettoyées (quelques minutes, environ 70 Mo). Les lancements suivants réutilisent les fichiers locaux de `data/`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Données

### 1.1 Sources

| Source | Contenu | Accès |
| :--- | :--- | :--- |
| **API PRIM** (Île-de-France Mobilités), temps réel | Passages de chaque train des RER A, B et D à chaque gare : horaire prévu, horaire réel ou estimé, retard, statut (supprimé ou non) | Collecté toutes les 2 minutes par notre collecteur (`bot_collect/`), déployé sur SSP Cloud depuis le 02/10/2026 |
| **API PRIM**, messages d'information | Messages officiels de perturbation des lignes (causes d'incidents) | Même collecteur |
| **Open data IDFM** | Coordonnées géographiques des gares | `src/coordonnees_gares.py` → `data/gares_coords.csv` |

Les données collectées sont stockées au format Parquet, une partition par jour, dans le bucket S3 **public** `gpichonlegroux` du SSP Cloud : elles sont donc accessibles par tous, sans identifiant.

### 1.2 Pipeline de nettoyage

1. `src/lecture.py` : synchronisation incrémentale du bucket S3 vers `data/raw/` (une ligne = un train, ses arrêts sont stockés en JSON dans la colonne `stops`).
2. `src/nettoyage.py` (avec DuckDB) :
   - éclatement du JSON : **une ligne = un arrêt d'un train dans une gare** ;
   - conservation des **tronçons centraux** (A : *Charles de Gaulle-Étoile* → *Nation* ; B : *Denfert-Rochereau* → *Aulnay-sous-Bois* ; D : *Gare de Lyon* → *Saint-Denis*) et uniformisation du nom *Châtelet-Les Halles* ;
   - conversion des horaires en heure de Paris ;
   - suppression des retards aberrants (hors de $[-15, 180]$ minutes) ;
   - création des variables d'analyse : `retard_min`, `delay_added_sec` (retard pris depuis l'arrêt précédent), `est_supprime`, `zone_tunnel` (Châtelet ou Gare du Nord), `heure_pointe`, `tranche_horaire`.

Le détail, avec le parcours d'un train d'exemple étape par étape, est dans `docs/pipeline_donnees.md`.

In [ ]:
from src.nettoyage import charger_donnees_propres

# Télécharge (si besoin) et nettoie les données, puis les charge
df_a, df_b, df_d, df_inc = charger_donnees_propres()

### 1.3 Description des données

In [ ]:
lignes = {"RER A": df_a, "RER B": df_b, "RER D": df_d}
apercu = pd.DataFrame({
    "Début": [d["date"].min().date() for d in lignes.values()],
    "Fin": [d["date"].max().date() for d in lignes.values()],
    "Gares du tronçon central": [d["stop_name"].nunique() for d in lignes.values()],
    "Trains": [len(d.drop_duplicates(["date", "journey_id"])) for d in lignes.values()],
    "Arrêts observés": [len(d) for d in lignes.values()],
}, index=lignes.keys())
apercu

In [ ]:
# Une ligne du tableau = un arrêt d'un train dans une gare du tronçon central
colonnes = ["date", "journey_id", "direction", "stop_name", "aimed_time", "expected_time",
            "retard_min", "delay_added_sec", "est_supprime", "tranche_horaire"]
df_b[colonnes].head()

In [ ]:
print(f"{len(df_inc)} messages d'incidents collectés")
df_inc[["line", "first_seen", "short_message"]].head()

Le RER B a le plus d'observations (12 gares sur son tronçon central). Le RER D en a 4 à 5 fois moins : seulement 5 gares centrales, et une collecte qui commence le 07/10. Les comparaisons se font donc toujours **en moyennes ou en proportions**, et l'axe 1 refait les calculs à **Châtelet**, seule gare commune aux trois lignes.

## 2. Axe 1 — Constat : B et D sont-ils plus en retard que A ?

Avant d'étudier la contagion, on vérifie qu'il existe bien un problème propre aux lignes qui partagent le tunnel. Le script `src/Axe1_constat_naif.py` compare les trois lignes sur :

- la **distribution des retards** : moyenne, médiane, et surtout 90e et 95e centiles, qui mesurent la violence des perturbations que la moyenne écrase ;
- le **retard ajouté à chaque arrêt**, dans le tunnel et en dehors, puis à l'entrée et à la sortie du tunnel ;
- l'effet de l'**heure de pointe** ;
- la part de **trains supprimés ou limités** en parcours ;
- le **retard moyen jour par jour**.

Les arrêts supprimés sont retirés des calculs de retard, car IDFM leur attribue un retard de 0.

In [ ]:
%run src/Axe1_constat_naif.py

### Interprétation

*Les chiffres cités ici correspondent aux données du 02/10 au 26/11/2026.*

- **B et D sont nettement plus en retard que A.** Le retard moyen est d'environ 3 minutes sur B et D contre 1,3 minute sur A. L'écart est encore plus net sur les retards graves : le 95e centile atteint environ 13 minutes sur B et D, contre 7 sur A.
- **Ce n'est pas un effet de longueur de tronçon** : on retrouve le même écart à Châtelet, la gare commune aux trois lignes.
- **Ce n'est pas non plus dû à quelques journées catastrophiques** : un jour ordinaire (médiane des retards journaliers), B et D sont 2 à 3 fois plus en retard que A. En revanche, B et D connaissent chacun environ 8 « journées noires » (plus de 5 minutes de retard moyen), contre aucune pour A.
- **Le tunnel est le lieu où le retard se crée.** Dans Paris, le RER A *rattrape* environ 11 secondes à chaque arrêt, alors que B et D en *prennent* respectivement 27 et 41 dans le tunnel, et près de 60 à 70 secondes en pointe du soir. Ce retard se prend surtout **à la sortie du tunnel** : environ 45 secondes (B) et 65 secondes (D) par arrêt. C'est la signature d'un goulot d'étranglement.
- **L'heure de pointe pèse deux fois plus** sur B et D que sur A.
- **La régulation passe par des trains coupés** : environ 2 % des trains du B et 3,5 % de ceux du D sont limités en parcours, contre 0,2 % pour le A.

**Conclusion de l'axe 1 :** les lignes qui partagent le tunnel sont plus en retard, plus souvent et plus fortement, et ce retard se forme précisément dans le tunnel. Cela reste une corrélation : B et D diffèrent aussi de A par leur longueur, leurs branches et leur exploitation. L'axe 2 teste directement la propagation.

## 3. Axe 2 — Contagion : les retards de B et D sont-ils liés ?

Si le tunnel transmet les perturbations, les retards de B et D doivent **évoluer ensemble**, et un retard de B doit **précéder** un retard de D. Le RER A, qui passe aussi à Châtelet mais sur ses propres voies, sert de **placebo**.

Le script `src/axe2_contagion.py` travaille sur le **retard moyen par fenêtre de 15 minutes à Châtelet** et calcule :

- les **corrélations** entre lignes, puis les mêmes corrélations une fois retiré l'effet commun de l'heure de la journée (*résidus*) ;
- une **analyse de décalage** avec `.shift()` : le retard de B à l'instant $t$ est-il lié au retard de D à $t + 15$, $t + 30$ minutes ?
- le retard de chaque ligne **selon l'état de l'autre** (fluide, ralenti, perturbé) ;
- la **résilience** : temps pour revenir sous 3 minutes de retard après un pic au-dessus de 10 minutes.

In [ ]:
%run src/axe2_contagion.py

### Interprétation

*Chiffres du 02/10 au 26/11/2026.*

- **Les retards de B et D évoluent ensemble.** À Châtelet, la corrélation B–D vaut environ 0,26, contre 0,08 pour B–A et 0,13 pour D–A. Le lien est modeste en valeur absolue, mais **trois fois plus fort** qu'avec la ligne témoin.
- **Ce n'est pas seulement l'heure de pointe** : une fois retiré l'effet commun de l'heure de la journée, la corrélation B–A tombe à 0,02, alors que B–D reste à 0,22.
- **Le retard de B précède celui de D** : la corrélation est maximale lorsqu'on compare B à l'instant $t$ avec D à $t + 15$ minutes.
- **L'effet domino est réciproque.** Quand B est perturbé (plus de 10 minutes de retard), le retard de D passe d'environ 2,3 à 7,4 minutes, tandis que A reste stable vers 1 minute. Inversement, quand D est perturbé, le retard de B monte à environ 7 minutes, contre 1,6 pour A. La contagion est plus forte dans le sens Sud (corrélation 0,29 contre 0,12 vers le Nord).
- **Résilience** : après un pic, B met en médiane environ 150 minutes à revenir à la normale, contre environ 70 minutes pour A. D récupère presque aussi vite que A, mais B et D subissent 3 à 4 fois plus de pics (33 et 43 contre 12).

**Conclusion de l'axe 2 :** les données sont cohérentes avec une propagation des perturbations entre B et D par le tunnel commun, et le placebo A ne montre pas ce lien. Les retards du B, en particulier, sont plus persistants.

## 4. Axe 3 — Détection automatique des principaux incidents

Le script `src/axe3_incidents.py` repère automatiquement les **épisodes de crise** dans le tunnel, par tranches de 15 minutes à Châtelet entre 6h30 et 21h30. Une tranche est en crise si :

- le retard moyen de B ou D dépasse 10 minutes ;
- **ou** le nombre de trains B + D tombe sous 50 % du débit habituel du créneau, avec plus de 6 minutes de retard ;
- **ou** plus d'un quart des passages B et D sont supprimés.

Les tranches proches (moins de 45 minutes d'écart) sont regroupées en épisodes, classés selon un score qui combine durée et intensité. Chaque épisode est croisé avec les **messages officiels d'incidents** pour en retrouver la cause, puis le plus important est étudié en détail, avec notamment un **diagramme de Marey** (trains en fonction du temps et de la gare).

In [ ]:
%run src/axe3_incidents.py

### Interprétation

*Chiffres du 02/10 au 26/11/2026.*

- **Les plus gros incidents durent longtemps et réduisent fortement le nombre de trains.** Les 8 principaux épisodes détectés durent de 2 heures à près de 8 heures (31/10, de 6h30 à 14h15). Pendant le pire d'entre eux, le nombre de trains B + D dans le tunnel tombe à 15 % du débit habituel, et jusqu'à 75 % des passages sont supprimés.
- **La plupart touchent B et D en même temps** : dans 6 épisodes sur 8, les deux lignes dépassent 10 minutes de retard moyen à Châtelet, souvent 20 à 50 minutes. Pendant ce temps, le RER A reste presque toujours sous 5 minutes dans la même gare. Sa seule hausse notable (environ 10 minutes, le 26/10) correspond à une panne électrique, qui touche probablement plusieurs lignes.
- **Les causes sont variées** : incidents d'exploitation, incidents affectant les voies, panne électrique, signal d'alarme, intervention des forces de l'ordre. Une perturbation née sur une ligne ou dans le tunnel se retrouve donc sur les deux lignes.
- **Il y a aussi des contre-exemples.** Le 15/10, un incident propre au RER D (réseau SNCF) fait monter D à près de 50 minutes, alors que B reste sous 8 minutes. Le 19/10 au soir, c'est l'inverse. La contagion n'est donc pas systématique : elle dépend probablement de l'endroit où survient l'incident.
- **Le zoom sur l'incident du 21/10** (incident affectant les voies, en pointe du matin) montre le mécanisme. Vers 7h15, le nombre de trains B + D passe d'environ 14 à 7 par quart d'heure. D atteint plus de 50 minutes de retard et B, entraîné, environ 15 minutes. D revient à la normale vers 9h, alors que B connaît un second pic vers 9h30 : le retard du B est plus persistant, comme l'avait montré l'axe 2.
- **Le diagramme de Marey du même jour** montre les arrêts supprimés au sud de Châtelet entre 7h et 7h45 : les trains du B sont limités en parcours et font demi-tour avant le tunnel. C'est le mode de régulation déjà repéré dans l'axe 1.

**Conclusion de l'axe 3 :** les épisodes de crise les plus graves touchent le plus souvent B et D ensemble, avec une chute du nombre de trains dans le tunnel. A, pendant ce temps, reste stable.

## 5. Axe 4 — Modélisation économétrique : contagion et effet tunnel

Les corrélations de l'axe 2 et les écarts moyens de l'axe 1 peuvent encore mélanger la congestion horaire commune et les spécificités des lignes. Le script `src/regression.py` estime deux familles de modèles économétriques (`statsmodels`, MCO) pour tester la robustesse du mécanisme :

1. **Contagion dynamique à Châtelet** (fenêtres de 15 minutes, erreurs-types **Newey-West HAC** à 4 retards pour tenir compte de l'autocorrélation) :
   - modèle à retards échelonnés : $D_t \sim B_t + B_{t-15} + B_{t-30} + \text{heure} + \text{week-end}$ ;
   - contrôle d'inertie propre : ajout de $D_{t-15}$ dans les variables explicatives ;
   - modèle symétrique $D \to B$ et **test placebo** sur le RER A ($B \to A$).
2. **Différence-en-Différences sur le retard ajouté par arrêt** (`delay_added_sec`, hors 1er arrêt de chaque train, erreurs-types robustes **HC3**) :
   - $\text{delay\_added\_sec} \sim \text{tunnel} \times \mathbf{1}_B + \text{tunnel} \times \mathbf{1}_D + \text{tranche\_horaire}$, où `tunnel` vaut 1 à Châtelet et Gare du Nord.

In [ ]:
%run src/regression.py

### Interprétation

*Chiffres du 02/10 au 26/11/2026.*

- **La contagion $B \to D$ est significative.** À Châtelet, une minute de retard supplémentaire sur le RER B est associée à **+0,165 minute** sur le RER D au même quart d'heure ($p = 0,005$) et **+0,148 minute** 15 minutes plus tard ($p = 0,002$).
- **L'effet ne s'explique pas uniquement par l'inertie du D.** Lorsqu'on ajoute le retard passé du RER D ($D_{t-15}$) comme contrôle, le retard décalé du B ($B_{t-15}$) reste positif et significatif (**+0,081**, $p = 0,027$).
- **Le test placebo sur le RER A est nul.** La même spécification appliquée à $B \to A$ donne un coefficient instantané de **+0,002** ($p = 0,832$) : le RER A, sur ses propres voies à Châtelet, ne réagit pas aux perturbations du B.
- **La contagion est réciproque.** Dans le sens $D \to B$, un retard de D à $t - 15$ minutes entraîne un retard significatif sur B (**+0,115** min/min, $p < 0,001$).
- **Le tunnel crée un surcoût net de retard à chaque arrêt.** En Différence-en-Différences par rapport au RER A (qui rattrape du retard à Châtelet), le passage dans le tronçon partagé ajoute **+39,2 s** par arrêt sur le RER B et **+54,6 s** par arrêt sur le RER D.

**Conclusion de l'axe 4 :** en contrôlant l'heure, le week-end et l'inertie propre des retards, la contagion bidirectionnelle entre B et D reste significative alors que le placebo A est nul, et le tunnel engendre un surcoût spécifique de 39 à 55 secondes par arrêt.

## 6. Conclusion

### Réponse à la problématique

Les quatre axes vont dans le même sens :

1. **Constat (axe 1)** : B et D sont plus en retard que A, surtout lors des perturbations graves, et ce retard se crée dans le tunnel commun, particulièrement à sa sortie.
2. **Contagion (axe 2)** : les retards de B et D évoluent ensemble, le retard de B précède celui de D d'environ 15 minutes, et l'effet joue dans les deux sens. Le RER A, utilisé comme placebo, ne montre pas ce lien. B met aussi plus longtemps à revenir à la normale.
3. **Incidents (axe 3)** : 6 des 8 principaux épisodes de crise touchent B et D ensemble, avec une forte chute du nombre de trains dans le tunnel, alors que A reste stable. Quelques incidents restent toutefois propres à une seule ligne.
4. **Modélisation (axe 4)** : la contagion dynamique (erreurs-types HAC) est confirmée dans les deux sens avec un placebo nul sur le RER A, et la Différence-en-Différences estime un surcoût net du tunnel d'environ +39 s par arrêt sur B et +55 s sur D.

L'hypothèse d'un **effet domino dans le tunnel B / D** est donc **confortée par les données**.

### Limites

- **Corrélation n'est pas causalité** : B et D diffèrent aussi de A par leur longueur, leurs branches et leur exploitation (deux opérateurs, RATP et SNCF).
- **Les liens mesurés restent modérés** : une corrélation d'environ 0,26 montre une tendance, pas une transmission systématique de chaque retard.
- **Les données temps réel sont imparfaites** : la collecte du RER D commence plus tard, il existe des trous de collecte, et le retard des trains supprimés n'est pas renseigné.
- **Plusieurs seuils sont des choix** : 3, 5 et 10 minutes, fenêtres de 15 minutes, critères de crise. Les conclusions devraient être vérifiées avec d'autres valeurs.
- **La période est courte** (un peu moins de deux mois, en automne) et s'allonge à chaque mise à jour des données : les chiffres peuvent légèrement évoluer.

### Pistes

- Affiner la modélisation économétrique en distinguant les sens de circulation (Nord / Sud) et d'éventuels effets de seuil lors des épisodes de saturation.
- Rejouer les incidents minute par minute sur une **carte interactive** (application Streamlit, `app.py`).

### Usage de l'IA générative

Nous avons utilisé une IA générative (Claude) pour nous aider à structurer et simplifier une partie du code (notamment les axes 1 et 2) et la mise en forme des graphiques. Les analyses et leur interprétation ont été relues et validées par l'équipe.